# 2-Stage 딥페이크 음성 탐지 시스템
## Multi-Feature Extraction + Temporal Sequence Modeling

### 연구 배경 및 동기

기존 1-Stage 모델(AttentionAudioClassifier)은 각 오디오 파일에서 MFCC 13차원의
**시간축 평균(mean pooling)** 벡터 하나만을 추출하여 분류에 사용했다.
이 방식은 시간적 변화 패턴(temporal dynamics)을 완전히 소실시킨다.

딥페이크 음성과 실제 음성의 핵심 차이는 **시간축 상의 미세한 불일치**에 있다:
- TTS 음성: 프로소디(운율)의 부자연스러운 전이, 포먼트 궤적의 비연속성
- Voice Conversion: 변환 과정에서 발생하는 프레임 간 불연속적 아티팩트
- 실제 음성: 자연스러운 코아티큘레이션(co-articulation)과 연속적 스펙트럴 변이

### 2-Stage 아키텍처 설계 철학

**Stage 1 (Feature Extractor):**
기존 학습된 AttentionAudioClassifier의 분류 헤드를 제거하고,
SE-Attention이 적용된 중간 표현(256차원 벡터)을 프레임 단위로 추출한다.
추가로, MFCC 외에 LFCC, Mel-Spectrogram, Spectral Contrast, Delta/Delta-Delta 등
다양한 음향 특징을 병렬로 추출하여 멀티-뷰 표현을 구성한다.

**Stage 2 (Temporal Classifier):**
프레임 단위 임베딩 시퀀스를 Transformer Encoder 또는 BiLSTM에 입력하여
시간적 패턴을 학습한다. 참고 논문:
- Xie et al. (2024) EAT: ResNet + Transformer Encoder + BiLSTM 결합
- LSTM-AE-DRDE (2025): Attention-enhanced LSTM with contrastive learning
- DK-CAST (2025): Multi-level supervision including embeddings and phoneme posteriors
- DLSA (2025): MFCC + CQT 2-stage MHA fusion, ASVspoof 2019 LA EER 0.68%

### 참고 문헌
1. Todisco et al., "ASVspoof 2019: Future horizons in spoofed and fake audio detection," 2019
2. Xie et al., "EAT: ResNet + Transformer + BiLSTM framework," 2024
3. Zaman et al., "Hybrid transformer architectures with diverse audio features," IEEE Access, 2024
4. Zhang et al., "Audio deepfake detection: what has been achieved and what lies ahead," Sensors, 2025
5. LSTM-AE-DRDE, "LSTM autoencoder with dynamic residual encoding," Scientific Reports, 2025
6. Shaaban, "Audio Deepfake Detection Using Deep Learning (Siamese CNN)," Engineering Reports, 2025
7. DLSA, "Spoof detection with dynamic learnable sparse attention and tri-modal fusion," PLOS ONE, 2025
8. DK-CAST, "Dynamic knowledge condensation with audio-selective transformer," Discover Computing, 2025
9. Deepfake voice detection with E2E transformer and acoustic feature fusion, Electronics, 2025


---
## 셀 1: 라이브러리 임포트 및 전역 설정

In [1]:
import os
import sys
import math
import copy
import warnings
import numpy as np
import librosa
import soundfile as sf
from pathlib import Path
from glob import glob
from tqdm import tqdm
from collections import Counter, OrderedDict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import (
    Dataset, DataLoader, WeightedRandomSampler, TensorDataset
)
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, roc_auc_score
)
from imblearn.over_sampling import SMOTE

import matplotlib.pyplot as plt
import seaborn as sns
import joblib

warnings.filterwarnings('ignore')

In [2]:


# ==============================================================
# 전역 설정
# ==============================================================
BASE_DIR = r'C:\Users\ipl\Desktop\test\DATASET'

# 오디오 전처리 파라미터
SAMPLE_RATE = 16000       # ASVspoof 표준 샘플링 레이트
SEGMENT_LENGTH = 4.0
OVERLAP = 0.25

# 특징 추출 파라미터
N_MFCC = 13               # MFCC 계수 (ASVspoof 표준)
N_LFCC = 20               # LFCC 계수 (고주파 해상도가 더 높으므로 MFCC보다 많이 사용)
N_MELS = 40               # Mel-Spectrogram 필터뱅크 수
N_FFT = 512               # FFT 윈도우 (32ms at 16kHz)
HOP_LENGTH = 160           # 홉 길이 (10ms at 16kHz, ASVspoof 표준)
WIN_LENGTH = 400           # 윈도우 길이 (25ms at 16kHz)

# 시퀀스 모델링 파라미터
MAX_SEQ_LEN = 400         # 최대 프레임 수 (4초 / 10ms = 400)
FRAME_EMBEDDING_DIM = 256 # Stage 1 출력 임베딩 차원

# 학습 파라미터
BATCH_SIZE = 64
NUM_EPOCHS = 100
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 20

# 시드 고정
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA ver: {torch.version.cuda}")
    print(f"GPU mem: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")


PyTorch: 2.5.1
CUDA: True
GPU: NVIDIA GeForce RTX 4070 Ti
CUDA ver: 12.1
GPU mem: 11.99 GB


---
## 셀 2: 다중 음향 특징 추출 함수 정의

교수님 피드백: "MFCC 특징 외에 다른 특징들도 함께 고려해도 좋을 것 같아요"

각 특징이 포착하는 음향적 성질:

### MFCC (Mel-Frequency Cepstral Coefficients)
- **물리적 의미**: 인간 청각의 비선형 주파수 감도(mel scale)를 반영한 스펙트럼 포락선
- **딥페이크 탐지 관련성**: 보코더가 생성하는 멜 필터뱅크 출력의 과도한 평활화를 감지
- **한계**: 고주파 대역(4kHz 이상)의 해상도가 낮아 보코더 아티팩트 포착이 제한적

### LFCC (Linear-Frequency Cepstral Coefficients)
- **물리적 의미**: 선형 주파수 스케일의 필터뱅크 기반 캡스트럼 계수
- **딥페이크 탐지 관련성**: MFCC와 달리 고주파 대역을 균등하게 커버하여
  neural vocoder의 고주파 아티팩트(금속성 잡음, 앨리어싱)를 더 잘 포착
- **근거**: ASVspoof 2019/2021에서 LFCC가 MFCC보다 일관적으로 우수한 성능 기록

### Mel-Spectrogram
- **물리적 의미**: 시간-주파수 에너지 분포를 멜 스케일로 압축한 2D 표현
- **딥페이크 탐지 관련성**: 프레임 간 에너지 전이의 연속성/불연속성을 보존

### Spectral Contrast
- **물리적 의미**: 각 서브밴드에서 스펙트럼 피크와 밸리의 에너지 차이
- **딥페이크 탐지 관련성**: 실제 음성은 하모닉 구조로 높은 콘트라스트,
  보코더 합성 음성은 스펙트럼이 더 평탄하여 콘트라스트가 낮은 경향

### Delta 및 Delta-Delta (시간 미분 계수)
- **물리적 의미**: 특징의 1차/2차 시간 미분, 즉 변화 속도와 가속도
- **딥페이크 탐지 관련성**: TTS 음성의 부자연스럽게 매끄러운 전이 vs 실제 음성의 자연스러운 변동
- **근거**: ASVspoof 기본 시스템에서 MFCC + Delta + Delta-Delta 조합이 표준


In [ ]:
def compute_lfcc(y, sr, n_lfcc=N_LFCC, n_fft=N_FFT,
                 hop_length=HOP_LENGTH, win_length=WIN_LENGTH):
    # LFCC (Linear Frequency Cepstral Coefficients) 추출
    #
    # MFCC가 멜 스케일(로그 주파수)을 사용하는 반면,
    # LFCC는 선형 주파수 스케일의 필터뱅크를 사용한다.
    #
    # 이론적 배경:
    #   선형 필터뱅크는 모든 주파수 대역을 균등한 대역폭으로 커버한다.
    #   따라서 고주파 대역(4kHz~8kHz)의 해상도가 MFCC보다 높다.
    #   Neural vocoder(WaveNet, WaveRNN, HiFi-GAN 등)는 고주파 대역에서
    #   앨리어싱, 금속성 노이즈, 부자연스러운 감쇠 등의 아티팩트를 남긴다.
    #   LFCC는 이러한 고주파 아티팩트를 더 정밀하게 포착할 수 있다.
    #
    # 구현:
    #   1. STFT -> 파워 스펙트로그램
    #   2. 선형 주파수 간격 삼각 필터뱅크 적용 (멜 스케일 대신)
    #   3. 로그 에너지 계산
    #   4. DCT로 캡스트럴 계수 추출
    #
    # 반환: (n_lfcc, T) 형태의 LFCC 행렬

    from scipy.fft import dct

    # 파워 스펙트로그램
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length,
                            win_length=win_length)) ** 2

    # 선형 주파수 간격 필터뱅크 생성
    n_filters = n_lfcc * 2
    freq_bins = n_fft // 2 + 1
    fmin = 0
    fmax = sr / 2

    # 선형 간격 중심 주파수 (멜 필터뱅크와의 핵심 차이)
    center_freqs = np.linspace(fmin, fmax, n_filters + 2)
    bin_indices = np.floor((n_fft + 1) * center_freqs / sr).astype(int)

    # 삼각 필터뱅크 구성
    filterbank = np.zeros((n_filters, freq_bins))
    for i in range(n_filters):
        left = bin_indices[i]
        center = bin_indices[i + 1]
        right = bin_indices[i + 2]
        if center > left:
            filterbank[i, left:center] = (
                np.arange(left, center) - left
            ) / (center - left)
        if right > center:
            filterbank[i, center:right] = (
                right - np.arange(center, right)
            ) / (right - center)

    # 필터뱅크 적용 -> 로그 -> DCT
    filter_energies = np.dot(filterbank, S)
    log_energies = np.log(filter_energies + 1e-10)
    lfcc = dct(log_energies, type=2, axis=0, norm='ortho')[:n_lfcc]

    return lfcc


def extract_multi_features_per_frame(audio_path, sr=SAMPLE_RATE,
                                      max_frames=MAX_SEQ_LEN):
    # 하나의 오디오 파일에서 프레임 단위 다중 특징 벡터를 추출한다.
    #
    # 추출되는 특징 (각 프레임 t에 대해):
    #   - MFCC (13D): 멜 스케일 스펙트럼 포락선
    #   - MFCC Delta (13D): 1차 시간 미분 (변화 속도)
    #     -> TTS는 파라미터 보간으로 delta가 비정상적으로 매끄러움
    #   - MFCC Delta-Delta (13D): 2차 시간 미분 (변화 가속도)
    #     -> 실제 음성은 코아티큘레이션으로 복잡한 가속 패턴
    #   - LFCC (20D): 선형 주파수 캡스트럼 (고주파 해상도 강화)
    #   - Mel-Spectrogram (40D): 멜 필터뱅크 에너지
    #   - Spectral Contrast (7D): 서브밴드 피크-밸리 에너지 차이
    #     -> 실제 음성은 하모닉 구조로 높은 콘트라스트,
    #        보코더 음성은 스펙트럼이 더 평탄하여 낮은 콘트라스트
    #   합계: 106차원 프레임 벡터
    #
    # 반환: (T, 106) numpy array 또는 None

    try:
        y, _ = librosa.load(audio_path, sr=sr, mono=True)

        # 0.5초 미만 제외
        if len(y) < sr * 0.5:
            return None

        # MFCC: 멜 스케일 스펙트럼 포락선 (ASVspoof 표준 13차원)
        mfcc = librosa.feature.mfcc(
            y=y, sr=sr, n_mfcc=N_MFCC,
            n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH
        )  # (13, T)

        # MFCC Delta: 인접 프레임 변화율 (width=5: 전후 2프레임)
        mfcc_delta = librosa.feature.delta(mfcc, width=5)  # (13, T)

        # MFCC Delta-Delta: 변화율의 변화율 (가속도)
        mfcc_delta2 = librosa.feature.delta(mfcc, order=2, width=5)  # (13, T)

        # LFCC: 고주파 보코더 아티팩트 탐지용 선형 주파수 캡스트럼
        lfcc = compute_lfcc(y, sr)  # (20, T)

        # Mel-Spectrogram: 시간-주파수 에너지 분포
        mel_spec = librosa.feature.melspectrogram(
            y=y, sr=sr, n_mels=N_MELS,
            n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH
        )
        log_mel = librosa.power_to_db(mel_spec, ref=np.max)  # (40, T)

        # Spectral Contrast: 서브밴드 피크-밸리 에너지 차이
        # 하모닉 구조가 있는 실제 음성 = 높은 콘트라스트
        # 보코더 음성 = 스펙트럼이 더 평탄 = 낮은 콘트라스트
        spectral_contrast = librosa.feature.spectral_contrast(
            y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH,
            n_bands=6  # 6 서브밴드 + 1 밸리 = 7차원
        )  # (7, T)

        # 프레임 수 통일 (각 특징의 T가 약간 다를 수 있음)
        min_frames_count = min(
            mfcc.shape[1], lfcc.shape[1],
            log_mel.shape[1], spectral_contrast.shape[1]
        )

        mfcc = mfcc[:, :min_frames_count]
        mfcc_delta = mfcc_delta[:, :min_frames_count]
        mfcc_delta2 = mfcc_delta2[:, :min_frames_count]
        lfcc = lfcc[:, :min_frames_count]
        log_mel = log_mel[:, :min_frames_count]
        spectral_contrast = spectral_contrast[:, :min_frames_count]

        # 모든 특징 결합: (106, T)
        combined = np.concatenate([
            mfcc,               # 13: 멜 스케일 스펙트럼 포락선
            mfcc_delta,         # 13: 시간적 변화 속도
            mfcc_delta2,        # 13: 시간적 변화 가속도
            lfcc,               # 20: 선형 주파수 캡스트럼
            log_mel,            # 40: 멜 스펙트로그램 에너지
            spectral_contrast,  #  7: 서브밴드 피크-밸리 차이
        ], axis=0)

        # 전치: (T, 106) - 시퀀스 모델 입력 형식
        combined = combined.T

        # 시퀀스 길이 제한
        if combined.shape[0] > max_frames:
            combined = combined[:max_frames]

        return combined

    except Exception as e:
        print(f"특징 추출 오류 ({audio_path}): {e}")
        return None


# 특징 추출 함수 테스트
print("다중 특징 추출 함수 결과")
test_signal = np.random.randn(SAMPLE_RATE * 3)
sf.write(r'C:\Users\ipl\Desktop\test\DATASET\train\real\R_000000.wav', test_signal, SAMPLE_RATE)
test_features = extract_multi_features_per_frame(r'C:\Users\ipl\Desktop\test\DATASET\train\real\R_000000.wav')
if test_features is not None:
    print(f"  출력 형태: {test_features.shape}")
    print(f"  프레임 수 (T): {test_features.shape[0]}")
    print(f"  특징 차원 (D): {test_features.shape[1]}")
    feat_breakdown = "MFCC(13) + Delta(13) + Delta2(13) + LFCC(20) + Mel(40) + Contrast(7)"
    print(f"  특징 구성: {feat_breakdown} = {13+13+13+20+40+7}")


다중 특징 추출 함수 테스트...
  출력 형태: (301, 106)
  프레임 수 (T): 301
  특징 차원 (D): 106
  특징 구성: MFCC(13) + Delta(13) + Delta2(13) + LFCC(20) + Mel(40) + Contrast(7) = 106


In [7]:
import os

root_path = r"C:\Users\ipl\Desktop\test"

for root, dirs, files in os.walk(root_path):
    # 현재 폴더 기준 들여쓰기 레벨 계산
    level = root.replace(root_path, "").count(os.sep)
    indent = " " * 4 * level

    print(f"{indent}{os.path.basename(root)}/")

    sub_indent = " " * 4 * (level + 1)
    for file in files:
        if not file.lower().endswith(".wav"):  # WAV 파일 제외
            print(f"{sub_indent}{file}")

test/
    data_combine.ipynb
    deepfake_2stage_detection_ckpt3.ipynb
    deepfake_final.ipynb
    test.ipynb
    DATASET/
        test/
            fake/
            real/
        train/
            fake/
            real/
        val/
            fake/
            real/


---
## 셀 3: Stage 1 모델 정의 - 기존 AttentionAudioClassifier 기반 임베딩 추출기

교수님 피드백: "현재 모델을 통해 각 오디오 시퀀스를 벡터 형태로 만들고"

기존 학습된 AttentionAudioClassifier의 가중치를 재활용한다.
핵심: 분류 헤드(classifier)를 제거하고, SE-Attention이 적용된
중간 표현(256차원)을 각 프레임의 "음성 품질 임베딩"으로 사용한다.

이론적 근거:
1. **전이 학습**: 기존 모델이 학습한 판별적 표현을 시퀀스 모델 입력으로 활용
2. **SE 블록**: 이미 "딥페이크 탐지에 중요한 특징 채널"에 대한 어텐션을 학습
3. **2-Stage 분리**: 특징 추출과 시간적 모델링을 독립 최적화, 해석 가능성 향상

참고: DK-CAST (2025) 교사 모델 중간 표현 전이, Wav2Vec2 fine-tuning 접근법


In [4]:
# ================================================================
# Stage 1-A: 기존 모델 구조 정의 (가중치 로드용)
# ================================================================

class SEBlock(nn.Module):
    # Squeeze-and-Excitation 블록 (채널 어텐션, Hu et al., 2018)
    #
    # 각 채널의 글로벌 정보를 squeeze하고, 채널 간 상호의존성을 excitation으로 모델링.
    # 수식: z = GlobalAvgPool(x), s = sigmoid(W2 * ReLU(W1 * z)), output = x * s
    #
    # 딥페이크 탐지에서의 역할:
    #   MFCC 13개 계수 중 딥페이크 구분에 변별력 있는 계수(고차 캡스트럼 등)에
    #   자동으로 더 높은 가중치를 할당한다.

    def __init__(self, channels, reduction=8):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)

    def forward(self, x):
        y = F.relu(self.fc1(x))
        y = torch.sigmoid(self.fc2(y))
        return x * y


class AttentionAudioClassifier(nn.Module):
    # 기존 1-Stage ASV 기반 딥페이크 분류기 (원본 구조 보존)
    # Stage 1에서 가중치 로드 및 중간 표현 추출 용도

    def __init__(self, input_dim, num_classes=2):
        super().__init__()
        self.fc1 = nn.Sequential(
            nn.Linear(input_dim, 512), nn.BatchNorm1d(512),
            nn.ReLU(), nn.Dropout(0.4)
        )
        self.se1 = SEBlock(512)
        self.fc2 = nn.Sequential(
            nn.Linear(512, 384), nn.BatchNorm1d(384),
            nn.ReLU(), nn.Dropout(0.35)
        )
        self.se2 = SEBlock(384)
        self.fc3 = nn.Sequential(
            nn.Linear(384, 256), nn.BatchNorm1d(256),
            nn.ReLU(), nn.Dropout(0.3)
        )
        self.se3 = SEBlock(256)
        self.classifier = nn.Sequential(
            nn.Linear(256, 128), nn.BatchNorm1d(128),
            nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.se1(self.fc1(x))
        x = self.se2(self.fc2(x))
        x = self.se3(self.fc3(x))
        return self.classifier(x)


# ================================================================
# Stage 1-B: 임베딩 추출기 (분류 헤드 제거)
# ================================================================

class Stage1EmbeddingExtractor(nn.Module):
    # 기존 AttentionAudioClassifier에서 classifier 헤드를 제거하고,
    # SE-Attention이 적용된 256차원 중간 표현을 출력한다.
    #
    # 동작: MFCC (13D) -> FC(512)+SE -> FC(384)+SE -> FC(256)+SE -> 출력 (256D)
    #
    # 이 256차원 벡터의 의미:
    #   기존 모델이 Real/Fake/TTS 분류를 위해 학습한 판별적 공간의 표현.
    #   SE 블록이 적용되어 "딥페이크 탐지에 중요한 특징"이 강조된 상태.
    #   각 프레임에 대한 "음성 품질 임베딩"으로 기능한다.
    #
    # 이론적 근거:
    #   전이 학습(Transfer Learning) - 기존 학습된 판별적 표현을 시퀀스 모델 입력으로 활용
    #   DK-CAST (2025): 교사 모델 중간 표현을 학생 모델에 전달하는 것과 유사
    #   Wav2Vec2 fine-tuning: 사전학습 모델 상위 레이어만 미세조정하는 것과 유사

    def __init__(self, input_dim=13):
        super().__init__()
        self.fc1 = nn.Sequential(
            nn.Linear(input_dim, 512), nn.BatchNorm1d(512),
            nn.ReLU(), nn.Dropout(0.4)
        )
        self.se1 = SEBlock(512)
        self.fc2 = nn.Sequential(
            nn.Linear(512, 384), nn.BatchNorm1d(384),
            nn.ReLU(), nn.Dropout(0.35)
        )
        self.se2 = SEBlock(384)
        self.fc3 = nn.Sequential(
            nn.Linear(384, 256), nn.BatchNorm1d(256),
            nn.ReLU(), nn.Dropout(0.3)
        )
        self.se3 = SEBlock(256)

    def forward(self, x):
        # x: (B, 13) 또는 (B, T, 13) 형태
        # 3D 입력: (B, T, 13) -> 프레임별 독립 처리 -> (B, T, 256)
        if x.dim() == 3:
            B, T, D = x.shape
            x = x.reshape(B * T, D)
            x = self.se1(self.fc1(x))
            x = self.se2(self.fc2(x))
            x = self.se3(self.fc3(x))
            return x.reshape(B, T, -1)
        else:
            x = self.se1(self.fc1(x))
            x = self.se2(self.fc2(x))
            x = self.se3(self.fc3(x))
            return x

    @classmethod
    def from_pretrained(cls, checkpoint_path, input_dim=13, device='cpu'):
        # 기존 AttentionAudioClassifier 가중치에서 분류 헤드를 제외하고 전이
        extractor = cls(input_dim=input_dim).to(device)
        state_dict = torch.load(checkpoint_path, map_location=device)

        # classifier.* 가중치 제외
        filtered = OrderedDict()
        for key, val in state_dict.items():
            if not key.startswith('classifier'):
                filtered[key] = val

        extractor.load_state_dict(filtered, strict=False)
        extractor.eval()
        print(f"Stage 1 임베딩 추출기 로드 완료 ({checkpoint_path})")
        print(f"  전이 파라미터: {len(filtered)}개, 제외: {len(state_dict)-len(filtered)}개 (classifier)")
        return extractor


# 기존 모델 가중치 로드 시도
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PRETRAINED_PATH = 'best_asvspoof_model.pt'
if os.path.exists(PRETRAINED_PATH):
    stage1_extractor = Stage1EmbeddingExtractor.from_pretrained(
        PRETRAINED_PATH, input_dim=N_MFCC, device=device
    )
    USE_PRETRAINED_STAGE1 = True
else:
    print(f"경고: '{PRETRAINED_PATH}' 파일 없음. Stage 1 랜덤 초기화.")
    print("(기존 모델 학습 후 이 셀을 다시 실행하면 전이 학습 적용)")
    stage1_extractor = Stage1EmbeddingExtractor(input_dim=N_MFCC).to(device)
    USE_PRETRAINED_STAGE1 = False


경고: 'best_asvspoof_model.pt' 파일 없음. Stage 1 랜덤 초기화.
(기존 모델 학습 후 이 셀을 다시 실행하면 전이 학습 적용)


---
## 셀 4: 프레임 단위 특징 추출 파이프라인

두 종류의 프레임 벡터를 추출하여 결합한다:

**A. 다중 음향 특징 (106차원)** - 물리적 신호 성질 직접 반영

**B. Stage 1 임베딩 (256차원)** - 데이터 기반 판별적 패턴 (선택적)

**결합 전략: Late Concatenation (106 + 256 = 362차원)**

이론적 근거 (다중 뷰 학습):
- 음향 특징과 학습된 임베딩은 상보적(complementary) 정보를 제공
- DLSA (2025)에서 MFCC + CQT + raw waveform 3-modal fusion으로 EER 0.68% 달성
- 기존 모델이 없으면 106차원 음향 특징만 사용


In [ ]:


# 데이터 요약
print("\n" + "=" * 60)
print("시퀀스 데이터 요약")
print("=" * 60)
for split in ['train', 'val', 'test']:
    d = data_sequences[split]
    if d['sequences']:
        lens_arr = np.array(d['lengths'])
        print(f"  {split}: {len(d['sequences'])}개, "
              f"dim={d['sequences'][0].shape[1]}, "
              f"len min/mean/max={lens_arr.min()}/{lens_arr.mean():.0f}/{lens_arr.max()}")


프레임 단위 다중 특징 추출 시작

train split 처리 중...
    real (0): 38301개 파일


  train/real:  69%|██████▉   | 26492/38301 [33:37<11:47, 16.70it/s]  

특징 추출 오류 (C:\Users\ipl\Desktop\test\DATASET\train\real\R_026490.wav): 


    real 완료: 300개 유효, 1개 건너뜀, 체크포인트: checkpoint_seq_train_real_72a8e0b8_chunks/
    fake (1): 88812개 파일


    fake 완료: 811개 유효, 1개 건너뜀, 체크포인트: checkpoint_seq_train_fake_234ba9db_chunks/
  train 전체 완료: 1111개, 차원: 106
  총 건너뛴 파일: 2개 (손상/메모리 오류)

val split 처리 중...
    real (0): 4787개 파일


    real 완료: 787개 유효, 0개 건너뜀, 체크포인트: checkpoint_seq_val_real_6b6775b8_chunks/
    fake (1): 11101개 파일


    fake 완료: 101개 유효, 0개 건너뜀, 체크포인트: checkpoint_seq_val_fake_0294f946_chunks/
  val 전체 완료: 888개, 차원: 106

test split 처리 중...
    real (0): 4789개 파일


    real 완료: 789개 유효, 0개 건너뜀, 체크포인트: checkpoint_seq_test_real_e36cf7fe_chunks/
    fake (1): 11102개 파일


    fake 완료: 102개 유효, 0개 건너뜀, 체크포인트: checkpoint_seq_test_fake_7f4bfc7b_chunks/
  test 전체 완료: 891개, 차원: 106

시퀀스 데이터 요약
  train: 1111개, dim=106, len min/mean/max=85/318/400
  val: 888개, dim=106, len min/mean/max=77/367/400
  test: 891개, dim=106, len min/mean/max=86/368/400


---
## 셀 5: 시퀀스 데이터 정규화 및 DataLoader 구성

### 정규화 전략
각 특징 차원에 대해 **전체 학습 데이터의 모든 프레임**을 기준으로
평균과 분산을 계산하고, 모든 split에 동일하게 적용한다.

### 시퀀스 SMOTE의 한계와 대안
- 시퀀스 데이터는 길이가 가변이므로 SMOTE 직접 적용이 어려움
- 대신 **클래스 가중치** + **WeightedRandomSampler**로 불균형 해결
- 학습 시 **SpecAugment**(시간/특징 마스킹)으로 추가 증강


In [ ]:
# ==============================================================
# 시퀀스 데이터 정규화
# 각 특징 차원에 대해 전체 학습 프레임 기준으로 평균/분산 계산
# 학습 데이터에만 fit, val/test는 transform만 수행
# ==============================================================

print("시퀀스 정규화 중...")

all_train_frames = np.concatenate(
    data_sequences['train']['sequences'], axis=0
)
print(f"  학습 프레임 수: {all_train_frames.shape[0]:,}")
print(f"  특징 차원: {all_train_frames.shape[1]}")

seq_scaler = StandardScaler()
seq_scaler.fit(all_train_frames)

for split in ['train', 'val', 'test']:
    for i in range(len(data_sequences[split]['sequences'])):
        data_sequences[split]['sequences'][i] = seq_scaler.transform(
            data_sequences[split]['sequences'][i]
        )

joblib.dump(seq_scaler, 'seq_scaler_2stage.pkl')
print("  스케일러 저장: seq_scaler_2stage.pkl")
del all_train_frames

# ==============================================================
# Dataset 및 DataLoader 생성
# ==============================================================

train_dataset = DeepfakeSequenceDataset(
    data_sequences['train']['sequences'],
    data_sequences['train']['labels'],
    data_sequences['train']['lengths']
)
val_dataset = DeepfakeSequenceDataset(
    data_sequences['val']['sequences'],
    data_sequences['val']['labels'],
    data_sequences['val']['lengths']
)
test_dataset = DeepfakeSequenceDataset(
    data_sequences['test']['sequences'],
    data_sequences['test']['labels'],
    data_sequences['test']['lengths']
)

# 클래스 가중치 기반 WeightedRandomSampler
# 기존 모델에서 TTS recall이 가장 낮았으므로 TTS에 3배 가중치
train_labels = data_sequences['train']['labels']
class_counts = Counter(train_labels)
total = sum(class_counts.values())
class_weight_map = {
    0: total / (2 * class_counts.get(0, 1)),  # real
    1: total / (2 * class_counts.get(1, 1))  # fake
}}
sample_weights = [class_weight_map[l] for l in train_labels]
sampler = WeightedRandomSampler(
    weights=sample_weights, num_samples=len(sample_weights), replacement=True
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
    collate_fn=collate_sequences, num_workers=0,
    pin_memory=torch.cuda.is_available()
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE * 2, shuffle=False,
    collate_fn=collate_sequences, num_workers=0,
    pin_memory=torch.cuda.is_available()
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE * 2, shuffle=False,
    collate_fn=collate_sequences, num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print(f"\nDataLoader 생성:")
print(f"  Train: {len(train_loader)} batches (bs={BATCH_SIZE})")
print(f"  Val:   {len(val_loader)} batches")
print(f"  Test:  {len(test_loader)} batches")
print(f"  클래스 분포: {dict(class_counts)}")
print(f"  클래스 가중치: { {k: f'{v:.4f}' for k, v in class_weight_map.items()} }")


---
## 셀 6: Stage 2 모델 정의 - Transformer / BiLSTM / Hybrid 분류기

3가지 Stage 2 아키텍처를 구현하여 비교:

### A. Transformer Encoder
- Self-attention이 시퀀스 내 임의 위치 간 의존성을 직접 모델링
- 장거리 의존성 포착에 강점 (음성 시작부-중간부 불일치 등)
- 참고: Zaman et al. (2024) "Hybrid transformer architectures"

### B. BiLSTM
- 순방향/역방향 시간 정보를 모두 활용하여 문맥 의존적 표현 학습
- 양방향 코아티큘레이션 패턴 모델링 가능
- 참고: LSTM-AE-DRDE (2025) "attention-enhanced LSTM"

### C. Hybrid (Transformer + BiLSTM)
- Transformer의 글로벌 어텐션("어디서" 아티팩트 발생) +
  LSTM의 순차적 모델링("어떤 순서로" 아티팩트 전개)
- 참고: Xie et al. (2024) EAT, Petmezas et al. (2025) CNN-LSTM-Transformer

### 공통 구성 요소
- **Positional Encoding**: 사인/코사인 (Vaswani et al., 2017)
- **SpecAugment**: 시간/특징 마스킹으로 일반화 향상 (Park et al., 2019)
- **Attention Pooling**: 딥페이크 단서 프레임에 높은 가중치 자동 부여


In [ ]:
class PositionalEncoding(nn.Module):
    # 사인/코사인 위치 인코딩 (Vaswani et al., 2017)
    #
    # Transformer는 순서 정보가 없으므로 위치 정보를 명시적으로 주입.
    # 서로 다른 주파수의 사인/코사인 함수로 각 위치에 고유한 벡터 할당.
    #
    # 성질:
    #   PE(pos+k)는 PE(pos)의 선형 함수 -> 상대 위치 학습 용이
    #   학습 불필요한 결정론적 인코딩 (파라미터 추가 없음)
    #   시퀀스 길이에 독립적 (학습보다 긴 시퀀스에도 적용 가능)

    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class SpecAugment(nn.Module):
    # SpecAugment 스타일 데이터 증강 (Park et al., 2019, 학습 시에만 적용)
    #
    # 시간 마스킹: 일부 프레임을 0으로 -> 특정 시간대에 의존하지 않도록
    # 특징 마스킹: 일부 차원을 0으로 -> 특정 특징에 과적합 방지
    # 목적: 모델이 "부분적 정보만으로도 딥페이크를 탐지"하도록 강제

    def __init__(self, time_mask_param=30, freq_mask_param=10,
                 num_time_masks=2, num_freq_masks=2):
        super().__init__()
        self.time_mask_param = time_mask_param
        self.freq_mask_param = freq_mask_param
        self.num_time_masks = num_time_masks
        self.num_freq_masks = num_freq_masks

    def forward(self, x):
        if not self.training:
            return x
        x = x.clone()
        B, T, D = x.shape
        for _ in range(self.num_time_masks):
            t = torch.randint(0, min(self.time_mask_param, T), (1,)).item()
            t0 = torch.randint(0, max(T - t, 1), (1,)).item()
            x[:, t0:t0+t, :] = 0
        for _ in range(self.num_freq_masks):
            f = torch.randint(0, min(self.freq_mask_param, D), (1,)).item()
            f0 = torch.randint(0, max(D - f, 1), (1,)).item()
            x[:, :, f0:f0+f] = 0
        return x


class TransformerClassifier(nn.Module):
    # Transformer Encoder 기반 시퀀스 분류기
    #
    # 아키텍처:
    #   입력 (B,T,D) -> Linear Projection -> Positional Encoding
    #   -> N x Transformer Encoder Layer -> Attention Pooling -> Classification Head
    #
    # Transformer Encoder의 역할:
    #   Self-attention이 모든 프레임 쌍 간의 관계를 직접 모델링.
    #   시간적으로 멀리 떨어진 프레임 간의 불일치도 한 번에 포착 가능.
    #   예: 음성 시작부의 자연스러운 onset과 중간부의 부자연스러운
    #   포먼트 전이 사이의 불일치를 감지.
    #
    # Attention Pooling:
    #   학습 가능한 query 벡터로 시퀀스를 가중 평균.
    #   "딥페이크 단서가 있는 프레임"에 자동으로 높은 가중치 부여.
    #
    # 참고: Zaman et al. (2024) "Hybrid transformer architectures"

    def __init__(self, input_dim, num_classes=2, d_model=256,
                 nhead=8, num_layers=4, dim_feedforward=512, dropout=0.3):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, d_model), nn.LayerNorm(d_model),
            nn.ReLU(), nn.Dropout(dropout)
        )
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        self.spec_augment = SpecAugment(30, 20, 2, 2)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward, dropout=dropout,
            activation='gelu',  # GELU: Transformer에서 ReLU보다 우수
            batch_first=True,
            norm_first=True     # Pre-LN: Post-LN보다 학습 안정성 향상
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers,
            norm=nn.LayerNorm(d_model)
        )

        self.attention_pool = nn.Sequential(
            nn.Linear(d_model, d_model // 4), nn.Tanh(),
            nn.Linear(d_model // 4, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.LayerNorm(d_model // 2),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model // 2, d_model // 4), nn.LayerNorm(d_model // 4),
            nn.GELU(), nn.Dropout(dropout * 0.5),
            nn.Linear(d_model // 4, num_classes)
        )

    def _create_padding_mask(self, lengths, max_len):
        # 패딩 위치에 True, attention이 부여되지 않도록 마스킹
        mask = torch.arange(max_len, device=lengths.device).unsqueeze(0)
        return mask >= lengths.unsqueeze(1)

    def forward(self, x, lengths=None):
        B, T, D = x.shape
        x = self.spec_augment(x)
        x = self.input_proj(x)
        x = self.pos_encoder(x)

        padding_mask = None
        if lengths is not None:
            padding_mask = self._create_padding_mask(lengths, T)

        x = self.transformer_encoder(x, src_key_padding_mask=padding_mask)

        # Attention Pooling: 딥페이크 단서 프레임에 높은 가중치
        attn_weights = self.attention_pool(x)
        if padding_mask is not None:
            attn_weights = attn_weights.masked_fill(
                padding_mask.unsqueeze(-1), float('-inf'))
        attn_weights = F.softmax(attn_weights, dim=1)
        pooled = torch.sum(x * attn_weights, dim=1)

        return self.classifier(pooled)


class BiLSTMClassifier(nn.Module):
    # 양방향 LSTM 기반 시퀀스 분류기
    #
    # BiLSTM 이론적 근거:
    #   순방향: 과거 프레임의 맥락으로 현재 프레임 해석
    #   역방향: 미래 프레임의 맥락으로 현재 프레임 해석
    #   양방향 결합: 전후 문맥을 모두 고려한 표현
    #
    # 딥페이크 탐지 장점:
    #   TTS의 포먼트 궤적은 "미래 음소에 대한 준비"가 부족할 수 있음.
    #   BiLSTM은 양방향 코아티큘레이션 패턴을 모델링.
    #   Transformer보다 적은 파라미터로 순차적 패턴 포착 가능.
    #
    # 참고: LSTM-AE-DRDE (2025) "attention-enhanced LSTM"

    def __init__(self, input_dim, num_classes=2, hidden_dim=256,
                 num_layers=3, dropout=0.3):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.ReLU(), nn.Dropout(dropout)
        )
        self.spec_augment = SpecAugment(30, 20, 2, 2)

        self.lstm = nn.LSTM(
            input_size=hidden_dim, hidden_size=hidden_dim,
            num_layers=num_layers, batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.layer_norm = nn.LayerNorm(hidden_dim * 2)

        self.attention_pool = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim // 2), nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.LayerNorm(hidden_dim // 2),
            nn.GELU(), nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x, lengths=None):
        B, T, D = x.shape
        x = self.spec_augment(x)
        x = self.input_proj(x)

        if lengths is not None:
            lengths_cpu = lengths.cpu().clamp(max=T)
            packed = pack_padded_sequence(x, lengths_cpu, batch_first=True, enforce_sorted=True)
            lstm_out, _ = self.lstm(packed)
            lstm_out, _ = pad_packed_sequence(lstm_out, batch_first=True, total_length=T)
        else:
            lstm_out, _ = self.lstm(x)

        lstm_out = self.layer_norm(lstm_out)

        attn_weights = self.attention_pool(lstm_out)
        if lengths is not None:
            mask = torch.arange(T, device=x.device).unsqueeze(0) >= lengths.unsqueeze(1)
            attn_weights = attn_weights.masked_fill(mask.unsqueeze(-1), float('-inf'))
        attn_weights = F.softmax(attn_weights, dim=1)
        pooled = torch.sum(lstm_out * attn_weights, dim=1)

        return self.classifier(pooled)


class HybridTransformerLSTM(nn.Module):
    # Transformer + BiLSTM 하이브리드 분류기
    #
    # 아키텍처:
    #   입력 -> Projection -> PosEnc -> Transformer (글로벌 어텐션)
    #   -> BiLSTM (순차적 모델링) -> Attention Pooling -> Classification
    #
    # 하이브리드 설계 이론적 근거:
    #   1. Transformer: Self-Attention으로 장거리 의존성 포착
    #      -> "어디서" 아티팩트가 있는지 (위치 독립적 관계)
    #   2. BiLSTM: 순차적 패턴 추가 모델링
    #      -> "어떤 순서로" 아티팩트가 나타나는지
    #
    # 참고:
    #   Xie et al. (2024) EAT: ResNet + Transformer + BiLSTM
    #   Petmezas et al. (2025): CNN-LSTM-Transformer hybrid

    def __init__(self, input_dim, num_classes=2, d_model=256,
                 nhead=8, num_transformer_layers=2,
                 num_lstm_layers=2, dropout=0.3):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, d_model), nn.LayerNorm(d_model),
            nn.ReLU(), nn.Dropout(dropout)
        )
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        self.spec_augment = SpecAugment(30, 20, 2, 2)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 2, dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=num_transformer_layers,
            norm=nn.LayerNorm(d_model)
        )

        # BiLSTM: hidden = d_model//2, bidirectional -> 출력 = d_model
        self.lstm = nn.LSTM(
            input_size=d_model, hidden_size=d_model // 2,
            num_layers=num_lstm_layers, batch_first=True,
            bidirectional=True,
            dropout=dropout if num_lstm_layers > 1 else 0
        )
        self.layer_norm = nn.LayerNorm(d_model)

        self.attention_pool = nn.Sequential(
            nn.Linear(d_model, d_model // 4), nn.Tanh(),
            nn.Linear(d_model // 4, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.LayerNorm(d_model // 2),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model // 2, num_classes)
        )

    def forward(self, x, lengths=None):
        B, T, D = x.shape
        x = self.spec_augment(x)
        x = self.input_proj(x)
        x = self.pos_encoder(x)

        padding_mask = None
        if lengths is not None:
            padding_mask = torch.arange(T, device=x.device).unsqueeze(0) >= lengths.unsqueeze(1)

        x = self.transformer(x, src_key_padding_mask=padding_mask)

        if lengths is not None:
            lengths_cpu = lengths.cpu().clamp(max=T)
            packed = pack_padded_sequence(x, lengths_cpu, batch_first=True, enforce_sorted=True)
            lstm_out, _ = self.lstm(packed)
            lstm_out, _ = pad_packed_sequence(lstm_out, batch_first=True, total_length=T)
        else:
            lstm_out, _ = self.lstm(x)

        lstm_out = self.layer_norm(lstm_out)

        attn_weights = self.attention_pool(lstm_out)
        if padding_mask is not None:
            attn_weights = attn_weights.masked_fill(padding_mask.unsqueeze(-1), float('-inf'))
        attn_weights = F.softmax(attn_weights, dim=1)
        pooled = torch.sum(lstm_out * attn_weights, dim=1)

        return self.classifier(pooled)


# ==============================================================
# 모델 생성
# ==============================================================
FEAT_DIM = data_sequences['train']['sequences'][0].shape[1]
print(f"입력 특징 차원: {FEAT_DIM}")

models = {
    'Transformer': TransformerClassifier(
        input_dim=FEAT_DIM, num_classes=2, d_model=256,
        nhead=8, num_layers=4, dim_feedforward=512, dropout=0.3
    ),
    'BiLSTM': BiLSTMClassifier(
        input_dim=FEAT_DIM, num_classes=2,
        hidden_dim=256, num_layers=3, dropout=0.3
    ),
    'Hybrid': HybridTransformerLSTM(
        input_dim=FEAT_DIM, num_classes=2, d_model=256,
        nhead=8, num_transformer_layers=2, num_lstm_layers=2, dropout=0.3
    )
}

print("\n모델 파라미터 수 비교:")
print("-" * 50)
for name, model in models.items():
    n_params = sum(p.numel() for p in model.parameters())
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  {name:15s}: {n_params:>10,} total, {n_train:>10,} trainable")


---
## 셀 7: 학습 루프 - 다중 모델 학습 및 비교

### 학습 전략
1. **손실 함수**: CrossEntropyLoss + 클래스 가중치 (TTS 3배 강화)
2. **옵티마이저**: AdamW (weight decay가 적응적 학습률과 올바르게 결합)
3. **학습률 스케줄러**: CosineAnnealingWarmRestarts
   - 코사인 곡선으로 감소, 주기적 재설정으로 지역 최솟값 탈출
4. **그래디언트 클리핑**: max_norm=1.0 (RNN/Transformer 폭발 방지)
5. **조기 종료**: patience=20, 기준=Macro F1 (불균형에 강건)


In [ ]:
def train_model(model, train_loader, val_loader, model_name,
                num_epochs=NUM_EPOCHS, lr=LEARNING_RATE,
                weight_decay=WEIGHT_DECAY, patience=PATIENCE,
                device=device):
    # Stage 2 모델 학습 함수
    #
    # 학습 전략:
    #   손실: CrossEntropyLoss + 클래스 가중치
    #   옵티마이저: AdamW (weight decay가 적응적 학습률과 올바르게 결합)
    #   스케줄러: CosineAnnealingWarmRestarts
    #     - 코사인 곡선으로 학습률 감소, 주기적으로 재설정하여 지역 최솟값 탈출
    #   그래디언트 클리핑: max_norm=1.0 (RNN/Transformer 폭발 방지)
    #   조기 종료: patience 에폭 동안 개선 없으면 중단
    #   평가 기준: Macro F1 (클래스 불균형에서 정확도보다 적절한 지표)

    model = model.to(device)

    cw = torch.FloatTensor([1.0, 1.0]).to(device)
    criterion = nn.CrossEntropyLoss(weight=cw)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-6
    )

    history = {
        'train_loss': [], 'train_acc': [], 'val_acc': [],
        'val_f1': [], 'lr': []
    }
    best_val_f1 = 0
    best_val_acc = 0
    patience_counter = 0

    print(f"\n{'='*60}")
    print(f"{model_name} 학습 시작")
    print(f"{'='*60}")

    for epoch in range(num_epochs):
        # --- 학습 ---
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for batch_seqs, batch_labels, batch_lengths in train_loader:
            batch_seqs = batch_seqs.to(device)
            batch_labels = batch_labels.to(device)
            batch_lengths = batch_lengths.to(device)

            optimizer.zero_grad()
            outputs = model(batch_seqs, batch_lengths)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            train_total += batch_labels.size(0)
            train_correct += (predicted == batch_labels).sum().item()

        scheduler.step()
        train_acc = train_correct / train_total
        avg_loss = train_loss / len(train_loader)

        # --- 검증 ---
        model.eval()
        val_preds = []
        val_labels_list = []

        with torch.no_grad():
            for batch_seqs, batch_labels, batch_lengths in val_loader:
                batch_seqs = batch_seqs.to(device)
                batch_lengths = batch_lengths.to(device)
                outputs = model(batch_seqs, batch_lengths)
                _, predicted = torch.max(outputs, 1)
                val_preds.extend(predicted.cpu().numpy())
                val_labels_list.extend(batch_labels.numpy())

        val_acc = accuracy_score(val_labels_list, val_preds)
        val_f1 = f1_score(val_labels_list, val_preds, average='macro')
        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(avg_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        history['lr'].append(current_lr)

        save_path = f'best_2stage_{model_name.lower()}.pt'
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
            patience_counter = 0
            marker = "  [BEST]"
        else:
            patience_counter += 1
            marker = ""

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoch [{epoch+1:3d}/{num_epochs}] - "
                  f"Loss: {avg_loss:.4f} | Train: {train_acc:.4f} | "
                  f"Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f} | "
                  f"LR: {current_lr:.6f}{marker}")

        if patience_counter >= patience:
            print(f"\n  조기 종료: {epoch+1} 에폭")
            break

    print(f"\n  학습 완료! 최고 Val Acc: {best_val_acc:.4f}, F1: {best_val_f1:.4f}")
    return history, best_val_acc


# ==============================================================
# 모델 학습 실행
# ==============================================================
MODELS_TO_TRAIN = ['Hybrid', 'Transformer', 'BiLSTM']

all_histories = {}
all_best_accs = {}

for model_name in MODELS_TO_TRAIN:
    model = models[model_name]
    history, best_acc = train_model(
        model, train_loader, val_loader, model_name, device=device
    )
    all_histories[model_name] = history
    all_best_accs[model_name] = best_acc


---
## 셀 8: 테스트 세트 평가

평가 지표:
- 정확도 / Macro F1 / 클래스별 Precision, Recall, F1 / 혼동 행렬


In [ ]:
def evaluate_model(model, test_loader, model_name, device=device):
    # 테스트 세트 평가: 정확도, Macro F1, 클래스별 P/R/F1, 혼동 행렬

    save_path = f'best_2stage_{model_name.lower()}.pt'
    if os.path.exists(save_path):
        model.load_state_dict(torch.load(save_path, map_location=device))
    model = model.to(device)
    model.eval()

    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for batch_seqs, batch_labels, batch_lengths in test_loader:
            batch_seqs = batch_seqs.to(device)
            batch_lengths = batch_lengths.to(device)
            outputs = model(batch_seqs, batch_lengths)
            probs = F.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(batch_labels.numpy())
            all_probs.extend(probs.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    accuracy = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    report = classification_report(
        all_labels, all_preds, target_names=class_names, digits=4
    )
    cm = confusion_matrix(all_labels, all_preds)

    print(f"\n{'='*60}")
    print(f"{model_name} - 테스트 평가")
    print(f"{'='*60}")
    print(f"  정확도: {accuracy:.4f}")
    print(f"  Macro F1: {macro_f1:.4f}")
    print(f"\n{report}")
    print(f"혼동 행렬:\n{cm}")

    return {
        'accuracy': accuracy, 'macro_f1': macro_f1,
        'preds': all_preds, 'labels': all_labels,
        'probs': all_probs, 'cm': cm, 'report': report
    }


all_results = {}
for model_name in MODELS_TO_TRAIN:
    model = models[model_name]
    results = evaluate_model(model, test_loader, model_name, device=device)
    all_results[model_name] = results


---
## 셀 9: 결과 시각화

1. 학습 곡선 비교 (Loss, Accuracy, F1, LR)
2. 모델별 혼동 행렬 (정규화)
3. 클래스별 성능 비교 바 차트 (Precision, Recall, F1)


In [ ]:
# ==============================================================
# 1. 학습 곡선 비교
# ==============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
colors = {'Transformer': '#2E86AB', 'BiLSTM': '#A23B72', 'Hybrid': '#F18F01'}

for name, hist in all_histories.items():
    axes[0, 0].plot(hist['train_loss'], label=name,
                     color=colors.get(name, 'gray'), linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training Loss', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

for name, hist in all_histories.items():
    c = colors.get(name, 'gray')
    axes[0, 1].plot(hist['train_acc'], label=f'{name} Train',
                     color=c, linewidth=2, linestyle='-')
    axes[0, 1].plot(hist['val_acc'], label=f'{name} Val',
                     color=c, linewidth=2, linestyle='--')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_title('Accuracy (Train vs Val)', fontweight='bold')
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(True, alpha=0.3)

for name, hist in all_histories.items():
    axes[1, 0].plot(hist['val_f1'], label=name,
                     color=colors.get(name, 'gray'), linewidth=2)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Macro F1')
axes[1, 0].set_title('Validation Macro F1', fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

for name, hist in all_histories.items():
    axes[1, 1].plot(hist['lr'], label=name,
                     color=colors.get(name, 'gray'), linewidth=2)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Learning Rate')
axes[1, 1].set_title('Learning Rate Schedule', fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_yscale('log')

plt.tight_layout()
plt.savefig('training_curves_2stage.png', dpi=150, bbox_inches='tight')
plt.show()

# ==============================================================
# 2. 모델별 혼동 행렬
# ==============================================================

n_models = len(all_results)
fig, axes = plt.subplots(1, n_models, figsize=(7 * n_models, 6))
if n_models == 1:
    axes = [axes]

for idx, (name, result) in enumerate(all_results.items()):
    cm = result['cm']
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[idx], vmin=0, vmax=1)
    axes[idx].set_title(
        f'{name}\nAcc: {result["accuracy"]:.4f}, F1: {result["macro_f1"]:.4f}',
        fontweight='bold')
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig('confusion_matrices_2stage.png', dpi=150, bbox_inches='tight')
plt.show()

# ==============================================================
# 3. 클래스별 성능 비교 바 차트
# ==============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics = ['precision', 'recall', 'f1-score']

for midx, metric in enumerate(metrics):
    model_names_list = list(all_results.keys())
    x = np.arange(len(class_names))
    width = 0.8 / len(model_names_list)

    for i, name in enumerate(model_names_list):
        report_dict = classification_report(
            all_results[name]['labels'], all_results[name]['preds'],
            target_names=class_names, output_dict=True
        )
        values = [report_dict[cn][metric] for cn in class_names]
        axes[midx].bar(x + i * width, values, width, label=name,
                       color=colors.get(name, 'gray'), alpha=0.85)

    axes[midx].set_xlabel('Class')
    axes[midx].set_ylabel(metric.capitalize())
    axes[midx].set_title(f'Class-wise {metric.capitalize()}', fontweight='bold')
    axes[midx].set_xticks(x + width * (len(model_names_list) - 1) / 2)
    axes[midx].set_xticklabels(class_names)
    axes[midx].legend()
    axes[midx].set_ylim(0, 1.05)
    axes[midx].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('classwise_performance_2stage.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 셀 10: 최종 성능 요약

In [ ]:
# ==============================================================
# 최종 성능 요약
# ==============================================================

print("=" * 70)
print("2-Stage 딥페이크 음성 탐지 - 최종 성능 요약")
print("=" * 70)

print(f"\n{'모델':<25} {'테스트 정확도':>15} {'Macro F1':>12}")
print("-" * 55)
for name, result in all_results.items():
    print(f"  {name:<23} {result['accuracy']:>15.4f} {result['macro_f1']:>12.4f}")

print("-" * 55)
print(f"\n클래스별 Recall 비교:")
print(f"{'모델':<25}", end="")
for cn in class_names:
    print(f" {cn:>10}", end="")
print()
print("-" * 55)

for name, result in all_results.items():
    report_dict = classification_report(
        result['labels'], result['preds'],
        target_names=class_names, output_dict=True
    )
    print(f"  {name:<23}", end="")
    for cn in class_names:
        print(f" {report_dict[cn]['recall']:>10.4f}", end="")
    print()

best_model_name = max(all_results, key=lambda k: all_results[k]['macro_f1'])
print(f"\n최고 모델: {best_model_name} "
      f"(Macro F1: {all_results[best_model_name]['macro_f1']:.4f})")

print(f"\n저장된 산출물:")
print(f"  seq_scaler_2stage.pkl")
for name in MODELS_TO_TRAIN:
    fname = f'best_2stage_{name.lower()}.pt'
    if os.path.exists(fname):
        size_mb = os.path.getsize(fname) / (1024 * 1024)
        print(f"  {fname:<35} ({size_mb:.1f} MB)")

if torch.cuda.is_available():
    print(f"\nGPU 메모리: "
          f"할당 {torch.cuda.memory_allocated(0)/1024**3:.2f} GB, "
          f"캐시 {torch.cuda.memory_reserved(0)/1024**3:.2f} GB")

print("\n" + "=" * 70)
print("완료!")
print("=" * 70)


---
## 셀 11: 추론 파이프라인

학습된 2-Stage 모델로 새 오디오 파일 예측:
1. 오디오 로드 -> 다중 음향 특징 (106D/frame)
2. (선택) Stage 1 임베딩 (256D/frame)
3. 특징 결합 + 정규화
4. Stage 2 시퀀스 모델 분류
5. 예측 결과 및 신뢰도 출력


In [ ]:
def load_2stage_pipeline(stage2_model_class, stage2_path,
                         stage1_path='best_asvspoof_model.pt',
                         scaler_path='seq_scaler_2stage.pkl',
                         mfcc_scaler_path='scaler_asvspoof.pkl',
                         input_dim=None, device=None):
    # 2-Stage 추론 파이프라인 로드
    #
    # 흐름: 오디오 -> 다중특징(106D) + Stage1임베딩(256D) -> 정규화
    #       -> Stage2 시퀀스모델 -> 예측
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    pipeline = {'device': device}

    # Stage 1 (선택적)
    if os.path.exists(stage1_path):
        pipeline['stage1'] = Stage1EmbeddingExtractor.from_pretrained(
            stage1_path, input_dim=N_MFCC, device=device
        )
        pipeline['stage1'].eval()
    else:
        pipeline['stage1'] = None

    # MFCC 스케일러
    if os.path.exists(mfcc_scaler_path):
        pipeline['mfcc_scaler'] = joblib.load(mfcc_scaler_path)
    else:
        pipeline['mfcc_scaler'] = None

    # 시퀀스 스케일러
    pipeline['seq_scaler'] = joblib.load(scaler_path)

    # Stage 2 모델
    if input_dim is None:
        input_dim = FEAT_DIM
    stage2 = stage2_model_class(input_dim=input_dim, num_classes=2)
    stage2.load_state_dict(torch.load(stage2_path, map_location=device))
    stage2 = stage2.to(device)
    stage2.eval()
    pipeline['stage2'] = stage2

    return pipeline


def predict_audio(audio_path, pipeline):
    # 새로운 오디오 파일에 대해 2-Stage 예측 수행
    #
    # 반환: 예측 클래스, 신뢰도, 클래스별 확률
    device = pipeline['device']

    # 1. 다중 음향 특징
    multi_feat = extract_multi_features_per_frame(audio_path)
    if multi_feat is None:
        return {'error': 'feature extraction failed'}

    # 2. Stage 1 임베딩 (선택적)
    if pipeline['stage1'] is not None:
        y, _ = librosa.load(audio_path, sr=SAMPLE_RATE)
        mfcc_raw = librosa.feature.mfcc(
            y=y, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
            n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH
        ).T

        T = min(multi_feat.shape[0], mfcc_raw.shape[0])
        multi_feat = multi_feat[:T]
        mfcc_raw = mfcc_raw[:T]

        if pipeline['mfcc_scaler'] is not None:
            mfcc_raw = pipeline['mfcc_scaler'].transform(mfcc_raw)

        with torch.no_grad():
            mfcc_tensor = torch.FloatTensor(mfcc_raw).unsqueeze(0).to(device)
            embedding = pipeline['stage1'](mfcc_tensor).squeeze(0).cpu().numpy()

        combined = np.concatenate([multi_feat, embedding[:T]], axis=1)
    else:
        combined = multi_feat

    # 3. 정규화
    combined = pipeline['seq_scaler'].transform(combined)

    # 4. Stage 2 예측
    with torch.no_grad():
        seq_tensor = torch.FloatTensor(combined).unsqueeze(0).to(device)
        lengths = torch.LongTensor([combined.shape[0]]).to(device)
        outputs = pipeline['stage2'](seq_tensor, lengths)
        probs = F.softmax(outputs, dim=1).cpu().numpy()[0]
        pred_class = np.argmax(probs)

    return {
        'file': os.path.basename(audio_path),
        'prediction': class_names[pred_class],
        'confidence': float(probs[pred_class]),
        'probabilities': {
            class_names[i]: float(probs[i]) for i in range(len(class_names))
        },
        'num_frames': combined.shape[0],
        'feature_dim': combined.shape[1]
    }


# 추론 테스트
print("2-Stage 추론 파이프라인 테스트")
print("-" * 50)

best_model_class = {
    'Transformer': TransformerClassifier,
    'BiLSTM': BiLSTMClassifier,
    'Hybrid': HybridTransformerLSTM
}[best_model_name]

best_model_path = f'best_2stage_{best_model_name.lower()}.pt'

if os.path.exists(best_model_path):
    pipeline = load_2stage_pipeline(
        best_model_class, best_model_path, input_dim=FEAT_DIM
    )
    print(f"파이프라인 로드 완료 (Stage 2: {best_model_name})")
    print(f"  Stage 1: {'사용' if pipeline['stage1'] is not None else '미사용'}")
    print(f"  입력 차원: {FEAT_DIM}")

    test_dir = os.path.join(BASE_DIR, 'test')
    if os.path.exists(test_dir):
        print("\n샘플 추론:")
        for cls_name in class_names:
            cls_dir = os.path.join(test_dir, cls_name)
            if os.path.exists(cls_dir):
                files = sorted(glob(os.path.join(cls_dir, '*.wav')))[:2]
                for f in files:
                    result = predict_audio(f, pipeline)
                    if 'error' not in result:
                        print(f"  [{cls_name:6s}] {result['file']:30s} -> "
                              f"{result['prediction']:6s} "
                              f"(conf: {result['confidence']:.4f})")
else:
    print(f"모델 파일 '{best_model_path}' 없음")

print("\n모든 작업 완료!")


---
## 시각화 A: 데이터셋 분포 분석

학습 전 데이터의 기본 통계와 음향 특징 분포를 시각적으로 확인한다.
이상치, 클래스 불균형, 특징 범위 등을 파악하여 전처리 적절성을 검증한다.


In [ ]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from glob import glob
from tqdm import tqdm
from scipy import stats

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

# ==============================================================
# 데이터셋 통계 수집
# ==============================================================

SAMPLE_LIMIT = 300  # 클래스당 샘플링 수 (전체 돌리면 너무 오래 걸림)

split_stats = {}
feature_samples = {'real': [], 'fake': []}

for split in ['train', 'val', 'test']:
    split_stats[split] = {}
    for cls in ['real', 'fake']:
        folder = os.path.join(BASE_DIR, split, cls)
        if not os.path.exists(folder):
            continue
        files = sorted(glob(os.path.join(folder, '*.wav')))
        durations = []
        for f in files:
            try:
                dur = librosa.get_duration(path=f)
                durations.append(dur)
            except Exception:
                pass
        split_stats[split][cls] = {
            'count': len(files),
            'durations': durations
        }

        # train에서만 음향 특징 샘플 수집
        if split == 'train' and len(feature_samples[cls]) < SAMPLE_LIMIT:
            sample_files = np.random.choice(files,
                min(SAMPLE_LIMIT, len(files)), replace=False)
            for f in tqdm(sample_files,
                          desc=f'  특징 수집 train/{cls}', leave=False):
                try:
                    y, sr = librosa.load(f, sr=16000, mono=True)
                    if len(y) < sr * 0.5:
                        continue
                    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13).mean(axis=1)
                    mel = librosa.feature.melspectrogram(
                        y=y, sr=sr, n_mels=40).mean(axis=1)
                    sc = librosa.feature.spectral_contrast(
                        y=y, sr=sr).mean(axis=1)
                    zcr = float(librosa.feature.zero_crossing_rate(y).mean())
                    rms = float(librosa.feature.rms(y=y).mean())
                    sc_mean = float(librosa.feature.spectral_centroid(
                        y=y, sr=sr).mean())
                    feature_samples[cls].append({
                        'mfcc': mfcc,
                        'mel': mel,
                        'sc': sc,
                        'zcr': zcr,
                        'rms': rms,
                        'spectral_centroid': sc_mean,
                        'duration': len(y) / sr
                    })
                except Exception:
                    pass

print('데이터 수집 완료')
for split in ['train', 'val', 'test']:
    for cls in ['real', 'fake']:
        if cls in split_stats.get(split, {}):
            d = split_stats[split][cls]
            durs = d['durations']
            print(f'  {split}/{cls}: {d["count"]:,}개  '
                  f'길이 {np.mean(durs):.2f}s (min {np.min(durs):.2f} '
                  f'max {np.max(durs):.2f})')


In [ ]:
# ==============================================================
# 그래프 1: 클래스 분포 + 음성 길이 분포
# ==============================================================

fig = plt.figure(figsize=(18, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

colors = {'real': '#2196F3', 'fake': '#F44336'}

# (1) 클래스별 파일 수 - split별 막대그래프
ax1 = fig.add_subplot(gs[0, 0])
splits = ['train', 'val', 'test']
x = np.arange(len(splits))
width = 0.35
for i, cls in enumerate(['real', 'fake']):
    counts = [split_stats[s].get(cls, {}).get('count', 0) for s in splits]
    bars = ax1.bar(x + i * width, counts, width,
                   label=cls, color=colors[cls], alpha=0.85)
    for bar, v in zip(bars, counts):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{v:,}', ha='center', va='bottom', fontsize=8)
ax1.set_title('Split별 클래스 파일 수')
ax1.set_xticks(x + width/2)
ax1.set_xticklabels(splits)
ax1.set_ylabel('파일 수')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# (2) 전체 클래스 비율 파이차트
ax2 = fig.add_subplot(gs[0, 1])
total_real = sum(split_stats[s].get('real', {}).get('count', 0) for s in splits)
total_fake = sum(split_stats[s].get('fake', {}).get('count', 0) for s in splits)
wedges, texts, autotexts = ax2.pie(
    [total_real, total_fake],
    labels=['real', 'fake'],
    autopct='%1.1f%%',
    colors=[colors['real'], colors['fake']],
    startangle=90,
    explode=(0.05, 0.05)
)
ax2.set_title(f'전체 클래스 비율\n(총 {total_real+total_fake:,}개)')

# (3) train 음성 길이 분포 히스토그램
ax3 = fig.add_subplot(gs[0, 2])
for cls in ['real', 'fake']:
    durs = split_stats['train'].get(cls, {}).get('durations', [])
    if durs:
        ax3.hist(durs, bins=50, alpha=0.6, label=cls,
                 color=colors[cls], density=True)
ax3.set_title('Train 음성 길이 분포')
ax3.set_xlabel('길이 (초)')
ax3.set_ylabel('밀도')
ax3.legend()
ax3.grid(alpha=0.3)

# (4) val 음성 길이 분포
ax4 = fig.add_subplot(gs[1, 0])
for cls in ['real', 'fake']:
    durs = split_stats['val'].get(cls, {}).get('durations', [])
    if durs:
        ax4.hist(durs, bins=40, alpha=0.6, label=cls,
                 color=colors[cls], density=True)
ax4.set_title('Val 음성 길이 분포')
ax4.set_xlabel('길이 (초)')
ax4.set_ylabel('밀도')
ax4.legend()
ax4.grid(alpha=0.3)

# (5) test 음성 길이 분포
ax5 = fig.add_subplot(gs[1, 1])
for cls in ['real', 'fake']:
    durs = split_stats['test'].get(cls, {}).get('durations', [])
    if durs:
        ax5.hist(durs, bins=40, alpha=0.6, label=cls,
                 color=colors[cls], density=True)
ax5.set_title('Test 음성 길이 분포')
ax5.set_xlabel('길이 (초)')
ax5.set_ylabel('밀도')
ax5.legend()
ax5.grid(alpha=0.3)

# (6) 클래스별 평균/표준편차 길이 비교
ax6 = fig.add_subplot(gs[1, 2])
for si, split in enumerate(splits):
    for ci, cls in enumerate(['real', 'fake']):
        durs = split_stats[split].get(cls, {}).get('durations', [])
        if durs:
            ax6.errorbar(
                si + ci * 0.3 - 0.15,
                np.mean(durs),
                yerr=np.std(durs),
                fmt='o', color=colors[cls],
                capsize=5, markersize=8,
                label=cls if si == 0 else ''
            )
ax6.set_title('Split별 평균 길이 (±std)')
ax6.set_xticks(range(len(splits)))
ax6.set_xticklabels(splits)
ax6.set_ylabel('길이 (초)')
ax6.legend()
ax6.grid(alpha=0.3)

plt.suptitle('데이터셋 분포 분석', fontsize=14, fontweight='bold', y=1.01)
plt.savefig('viz_01_dataset_distribution.png', bbox_inches='tight', dpi=120)
plt.show()
print('저장: viz_01_dataset_distribution.png')


In [ ]:
# ==============================================================
# 그래프 2: MFCC 계수별 분포 (real vs fake)
# ==============================================================

real_mfcc = np.array([s['mfcc'] for s in feature_samples['real']])  # (N, 13)
fake_mfcc = np.array([s['mfcc'] for s in feature_samples['fake']])

fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.flatten()

for i in range(13):
    ax = axes[i]
    ax.hist(real_mfcc[:, i], bins=40, alpha=0.6, density=True,
            color='#2196F3', label='real')
    ax.hist(fake_mfcc[:, i], bins=40, alpha=0.6, density=True,
            color='#F44336', label='fake')

    # KDE 곡선
    for data, color in [(real_mfcc[:, i], '#1565C0'),
                        (fake_mfcc[:, i], '#B71C1C')]:
        kde_x = np.linspace(data.min(), data.max(), 200)
        kde = stats.gaussian_kde(data)
        ax.plot(kde_x, kde(kde_x), color=color, linewidth=2)

    # t-검정 p값
    t_stat, p_val = stats.ttest_ind(real_mfcc[:, i], fake_mfcc[:, i])
    ax.set_title(f'MFCC {i+1}  p={p_val:.3f}'
                 + (' *' if p_val < 0.05 else ''), fontsize=9)
    ax.set_xlabel('계수값')
    ax.set_ylabel('밀도')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

# 나머지 subplot 숨기기
for i in range(13, len(axes)):
    axes[i].set_visible(False)

plt.suptitle('MFCC 계수별 real vs fake 분포\n(* p<0.05: 통계적으로 유의미한 차이)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('viz_02_mfcc_distribution.png', bbox_inches='tight', dpi=120)
plt.show()
print('저장: viz_02_mfcc_distribution.png')


In [ ]:
# ==============================================================
# 그래프 3: MFCC 입체 그래프 (3D scatter - 주요 3계수)
# ==============================================================

from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(20, 6))

# 3D scatter: MFCC 1,2,3
ax1 = fig.add_subplot(131, projection='3d')
n = min(len(real_mfcc), len(fake_mfcc), 200)
idx_r = np.random.choice(len(real_mfcc), n, replace=False)
idx_f = np.random.choice(len(fake_mfcc), n, replace=False)

ax1.scatter(real_mfcc[idx_r, 0], real_mfcc[idx_r, 1], real_mfcc[idx_r, 2],
            c='#2196F3', alpha=0.6, s=20, label='real')
ax1.scatter(fake_mfcc[idx_f, 0], fake_mfcc[idx_f, 1], fake_mfcc[idx_f, 2],
            c='#F44336', alpha=0.6, s=20, label='fake')
ax1.set_xlabel('MFCC 1')
ax1.set_ylabel('MFCC 2')
ax1.set_zlabel('MFCC 3')
ax1.set_title('MFCC 1-2-3 3D 분포')
ax1.legend(fontsize=8)

# 3D scatter: MFCC 4,5,6
ax2 = fig.add_subplot(132, projection='3d')
ax2.scatter(real_mfcc[idx_r, 3], real_mfcc[idx_r, 4], real_mfcc[idx_r, 5],
            c='#2196F3', alpha=0.6, s=20, label='real')
ax2.scatter(fake_mfcc[idx_f, 3], fake_mfcc[idx_f, 4], fake_mfcc[idx_f, 5],
            c='#F44336', alpha=0.6, s=20, label='fake')
ax2.set_xlabel('MFCC 4')
ax2.set_ylabel('MFCC 5')
ax2.set_zlabel('MFCC 6')
ax2.set_title('MFCC 4-5-6 3D 분포')
ax2.legend(fontsize=8)

# 3D scatter: MFCC 1,2 + RMS
ax3 = fig.add_subplot(133, projection='3d')
real_rms = np.array([s['rms'] for s in feature_samples['real']])
fake_rms = np.array([s['rms'] for s in feature_samples['fake']])
ax3.scatter(real_mfcc[idx_r, 0], real_mfcc[idx_r, 1], real_rms[idx_r],
            c='#2196F3', alpha=0.6, s=20, label='real')
ax3.scatter(fake_mfcc[idx_f, 0], fake_mfcc[idx_f, 1], fake_rms[idx_f],
            c='#F44336', alpha=0.6, s=20, label='fake')
ax3.set_xlabel('MFCC 1')
ax3.set_ylabel('MFCC 2')
ax3.set_zlabel('RMS Energy')
ax3.set_title('MFCC 1-2 + RMS 3D 분포')
ax3.legend(fontsize=8)

plt.suptitle('음향 계수 3D 입체 분포', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('viz_03_mfcc_3d.png', bbox_inches='tight', dpi=120)
plt.show()
print('저장: viz_03_mfcc_3d.png')


In [ ]:
# ==============================================================
# 그래프 4: 스펙트럼 특징 분포 (Mel, Spectral Contrast, ZCR, RMS)
# ==============================================================

real_mel  = np.array([s['mel'] for s in feature_samples['real']])
fake_mel  = np.array([s['mel'] for s in feature_samples['fake']])
real_sc   = np.array([s['sc'] for s in feature_samples['real']])
fake_sc   = np.array([s['sc'] for s in feature_samples['fake']])
real_zcr  = np.array([s['zcr'] for s in feature_samples['real']])
fake_zcr  = np.array([s['zcr'] for s in feature_samples['fake']])
real_sc_c = np.array([s['spectral_centroid'] for s in feature_samples['real']])
fake_sc_c = np.array([s['spectral_centroid'] for s in feature_samples['fake']])

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

# Mel-Spectrogram 평균 비교 (멜 빈별)
ax = axes[0, 0]
mel_bins = np.arange(40)
ax.plot(mel_bins, real_mel.mean(axis=0), color='#2196F3',
        linewidth=2, label='real')
ax.fill_between(mel_bins,
    real_mel.mean(axis=0) - real_mel.std(axis=0),
    real_mel.mean(axis=0) + real_mel.std(axis=0),
    alpha=0.2, color='#2196F3')
ax.plot(mel_bins, fake_mel.mean(axis=0), color='#F44336',
        linewidth=2, label='fake')
ax.fill_between(mel_bins,
    fake_mel.mean(axis=0) - fake_mel.std(axis=0),
    fake_mel.mean(axis=0) + fake_mel.std(axis=0),
    alpha=0.2, color='#F44336')
ax.set_title('Mel-Spectrogram 평균 에너지 (멜 빈별)')
ax.set_xlabel('Mel 빈 인덱스')
ax.set_ylabel('에너지 (dB)')
ax.legend()
ax.grid(alpha=0.3)

# Spectral Contrast 평균 비교
ax = axes[0, 1]
sc_bins = np.arange(7)
ax.plot(sc_bins, real_sc.mean(axis=0), 'o-', color='#2196F3',
        linewidth=2, markersize=8, label='real')
ax.fill_between(sc_bins,
    real_sc.mean(axis=0) - real_sc.std(axis=0),
    real_sc.mean(axis=0) + real_sc.std(axis=0),
    alpha=0.2, color='#2196F3')
ax.plot(sc_bins, fake_sc.mean(axis=0), 's-', color='#F44336',
        linewidth=2, markersize=8, label='fake')
ax.fill_between(sc_bins,
    fake_sc.mean(axis=0) - fake_sc.std(axis=0),
    fake_sc.mean(axis=0) + fake_sc.std(axis=0),
    alpha=0.2, color='#F44336')
ax.set_title('Spectral Contrast 서브밴드별 평균')
ax.set_xlabel('서브밴드')
ax.set_ylabel('Contrast 값')
ax.legend()
ax.grid(alpha=0.3)

# ZCR 분포
ax = axes[0, 2]
ax.hist(real_zcr, bins=40, alpha=0.6, density=True,
        color='#2196F3', label='real')
ax.hist(fake_zcr, bins=40, alpha=0.6, density=True,
        color='#F44336', label='fake')
ax.axvline(real_zcr.mean(), color='#1565C0', linestyle='--', linewidth=2)
ax.axvline(fake_zcr.mean(), color='#B71C1C', linestyle='--', linewidth=2)
ax.set_title(f'Zero Crossing Rate 분포\n'
             f'real: {real_zcr.mean():.4f}  fake: {fake_zcr.mean():.4f}')
ax.set_xlabel('ZCR')
ax.set_ylabel('밀도')
ax.legend()
ax.grid(alpha=0.3)

# RMS Energy 분포
ax = axes[0, 3]
ax.hist(real_rms, bins=40, alpha=0.6, density=True,
        color='#2196F3', label='real')
ax.hist(fake_rms, bins=40, alpha=0.6, density=True,
        color='#F44336', label='fake')
ax.axvline(real_rms.mean(), color='#1565C0', linestyle='--', linewidth=2)
ax.axvline(fake_rms.mean(), color='#B71C1C', linestyle='--', linewidth=2)
ax.set_title(f'RMS Energy 분포\n'
             f'real: {real_rms.mean():.4f}  fake: {fake_rms.mean():.4f}')
ax.set_xlabel('RMS')
ax.set_ylabel('밀도')
ax.legend()
ax.grid(alpha=0.3)

# Spectral Centroid 분포
ax = axes[1, 0]
ax.hist(real_sc_c, bins=40, alpha=0.6, density=True,
        color='#2196F3', label='real')
ax.hist(fake_sc_c, bins=40, alpha=0.6, density=True,
        color='#F44336', label='fake')
ax.set_title('Spectral Centroid 분포')
ax.set_xlabel('Hz')
ax.set_ylabel('밀도')
ax.legend()
ax.grid(alpha=0.3)

# Mel 히트맵: real vs fake 평균 차이
ax = axes[1, 1]
mel_diff = real_mel.mean(axis=0) - fake_mel.mean(axis=0)
ax.bar(np.arange(40), mel_diff,
       color=['#2196F3' if v > 0 else '#F44336' for v in mel_diff])
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Mel 에너지 차이 (real - fake)\n양수=real이 높음, 음수=fake가 높음')
ax.set_xlabel('Mel 빈')
ax.set_ylabel('에너지 차이 (dB)')
ax.grid(alpha=0.3)

# Violin plot: 주요 스칼라 특징 비교
ax = axes[1, 2]
data_violin = [
    real_zcr, fake_zcr
]
parts = ax.violinplot(data_violin, positions=[1, 2], showmeans=True)
parts['bodies'][0].set_facecolor('#2196F3')
parts['bodies'][1].set_facecolor('#F44336')
ax.set_xticks([1, 2])
ax.set_xticklabels(['real', 'fake'])
ax.set_title('ZCR Violin Plot')
ax.set_ylabel('ZCR')
ax.grid(alpha=0.3)

# Violin plot: RMS
ax = axes[1, 3]
parts = ax.violinplot([real_rms, fake_rms],
                      positions=[1, 2], showmeans=True)
parts['bodies'][0].set_facecolor('#2196F3')
parts['bodies'][1].set_facecolor('#F44336')
ax.set_xticks([1, 2])
ax.set_xticklabels(['real', 'fake'])
ax.set_title('RMS Energy Violin Plot')
ax.set_ylabel('RMS')
ax.grid(alpha=0.3)

plt.suptitle('스펙트럼 특징 분포 분석', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('viz_04_spectral_features.png', bbox_inches='tight', dpi=120)
plt.show()
print('저장: viz_04_spectral_features.png')


In [ ]:
# ==============================================================
# 그래프 5: MFCC 상관관계 히트맵 + 박스플롯
# ==============================================================

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# (1) real MFCC 상관관계 히트맵
ax = axes[0]
corr_real = np.corrcoef(real_mfcc.T)
im = ax.imshow(corr_real, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_title('Real MFCC 계수간 상관관계')
ax.set_xticks(range(13))
ax.set_yticks(range(13))
ax.set_xticklabels([f'M{i+1}' for i in range(13)], fontsize=8)
ax.set_yticklabels([f'M{i+1}' for i in range(13)], fontsize=8)
plt.colorbar(im, ax=ax)
for i in range(13):
    for j in range(13):
        ax.text(j, i, f'{corr_real[i,j]:.1f}',
                ha='center', va='center', fontsize=6,
                color='white' if abs(corr_real[i,j]) > 0.5 else 'black')

# (2) fake MFCC 상관관계 히트맵
ax = axes[1]
corr_fake = np.corrcoef(fake_mfcc.T)
im = ax.imshow(corr_fake, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_title('Fake MFCC 계수간 상관관계')
ax.set_xticks(range(13))
ax.set_yticks(range(13))
ax.set_xticklabels([f'M{i+1}' for i in range(13)], fontsize=8)
ax.set_yticklabels([f'M{i+1}' for i in range(13)], fontsize=8)
plt.colorbar(im, ax=ax)
for i in range(13):
    for j in range(13):
        ax.text(j, i, f'{corr_fake[i,j]:.1f}',
                ha='center', va='center', fontsize=6,
                color='white' if abs(corr_fake[i,j]) > 0.5 else 'black')

# (3) MFCC 박스플롯 비교
ax = axes[2]
all_mfcc = np.vstack([real_mfcc, fake_mfcc])
labels_box = ['real'] * len(real_mfcc) + ['fake'] * len(fake_mfcc)
positions_r = np.arange(13) * 2
positions_f = np.arange(13) * 2 + 0.8

bp1 = ax.boxplot([real_mfcc[:, i] for i in range(13)],
                 positions=positions_r, widths=0.6,
                 patch_artist=True,
                 boxprops=dict(facecolor='#90CAF9', alpha=0.7),
                 medianprops=dict(color='#1565C0', linewidth=2),
                 flierprops=dict(marker='o', markersize=2, alpha=0.3))
bp2 = ax.boxplot([fake_mfcc[:, i] for i in range(13)],
                 positions=positions_f, widths=0.6,
                 patch_artist=True,
                 boxprops=dict(facecolor='#EF9A9A', alpha=0.7),
                 medianprops=dict(color='#B71C1C', linewidth=2),
                 flierprops=dict(marker='o', markersize=2, alpha=0.3))
ax.set_title('MFCC 계수별 Box Plot (real vs fake)')
ax.set_xticks(positions_r + 0.4)
ax.set_xticklabels([f'M{i+1}' for i in range(13)], fontsize=8)
ax.set_ylabel('계수값')
ax.legend([bp1['boxes'][0], bp2['boxes'][0]], ['real', 'fake'])
ax.grid(axis='y', alpha=0.3)

plt.suptitle('MFCC 상관관계 및 분포 심층 분석',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('viz_05_mfcc_correlation.png', bbox_inches='tight', dpi=120)
plt.show()
print('저장: viz_05_mfcc_correlation.png')


In [ ]:
# ==============================================================
# 그래프 6: 샘플 파형 + 스펙트로그램 비교 (real vs fake)
# ==============================================================

import random

samples_to_show = 2
fig, axes = plt.subplots(4, samples_to_show * 2, figsize=(22, 14))

for col_base, cls in enumerate(['real', 'fake']):
    folder = os.path.join(BASE_DIR, 'train', cls)
    files = sorted(glob(os.path.join(folder, '*.wav')))
    chosen = random.sample(files, min(samples_to_show, len(files)))

    for s_idx, fpath in enumerate(chosen):
        col = col_base * samples_to_show + s_idx
        y, sr = librosa.load(fpath, sr=16000, mono=True)
        fname = os.path.basename(fpath)
        title_prefix = f'[{cls}] {fname[:20]}'

        # 파형
        t = np.linspace(0, len(y)/sr, len(y))
        axes[0, col].plot(t, y,
            color='#2196F3' if cls == 'real' else '#F44336',
            linewidth=0.5, alpha=0.8)
        axes[0, col].set_title(f'{title_prefix}\n파형', fontsize=8)
        axes[0, col].set_xlabel('시간 (초)')
        axes[0, col].set_ylabel('진폭')
        axes[0, col].grid(alpha=0.3)

        # Mel 스펙트로그램
        mel = librosa.feature.melspectrogram(
            y=y, sr=sr, n_mels=80, n_fft=512,
            hop_length=160, win_length=400)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        img = librosa.display.specshow(
            mel_db, sr=sr, hop_length=160,
            x_axis='time', y_axis='mel',
            ax=axes[1, col], cmap='magma')
        axes[1, col].set_title(f'Mel 스펙트로그램', fontsize=8)
        plt.colorbar(img, ax=axes[1, col], format='%+2.0f dB')

        # MFCC
        mfcc = librosa.feature.mfcc(
            y=y, sr=sr, n_mfcc=13, n_fft=512,
            hop_length=160, win_length=400)
        img2 = librosa.display.specshow(
            mfcc, sr=sr, hop_length=160,
            x_axis='time', ax=axes[2, col], cmap='coolwarm')
        axes[2, col].set_title(f'MFCC (13차원)', fontsize=8)
        axes[2, col].set_ylabel('MFCC 계수')
        plt.colorbar(img2, ax=axes[2, col])

        # Spectral Contrast
        sc = librosa.feature.spectral_contrast(
            y=y, sr=sr, n_fft=512, hop_length=160)
        img3 = librosa.display.specshow(
            sc, sr=sr, hop_length=160,
            x_axis='time', ax=axes[3, col], cmap='viridis')
        axes[3, col].set_title(f'Spectral Contrast', fontsize=8)
        axes[3, col].set_ylabel('서브밴드')
        plt.colorbar(img3, ax=axes[3, col])

plt.suptitle('Real vs Fake 샘플 음향 특징 시각화',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('viz_06_sample_features.png', bbox_inches='tight', dpi=120)
plt.show()
print('저장: viz_06_sample_features.png')


---
## 셀 12: 아키텍처 요약

```
                    2-Stage 딥페이크 음성 탐지 시스템
                    ================================

 [오디오 입력 (WAV)]
        |
        v
 +----------------------------------------------+
 |     다중 음향 특징 추출 (프레임 단위)           |
 |  MFCC (13D) + Delta (13D) + Delta2 (13D)     |
 |  LFCC (20D) + Mel-Spec (40D) + Contrast (7D) |
 |  합계: 106차원 프레임 벡터                      |
 +----------------------------------------------+
        |                    |
        |                    v
        |     +--------------------------------------+
        |     | Stage 1: SE-Attention 임베딩 추출기   |
        |     | (기존 모델 전이 학습)                  |
        |     | MFCC(13) -> FC+SE -> 256D 임베딩      |
        |     +--------------------------------------+
        |                    |
        v                    v
 +----------------------------------------------+
 |  프레임 특징 결합: 106D + 256D = 362D          |
 +----------------------------------------------+
        |
        v
 +----------------------------------------------+
 |     Stage 2: 시간적 시퀀스 모델링               |
 |  A. Transformer Encoder (글로벌 어텐션)        |
 |  B. BiLSTM (양방향 순차 모델링)                |
 |  C. Hybrid (Transformer + BiLSTM)             |
 +----------------------------------------------+
        |
        v
 +----------------------------------------------+
 |  Attention Pooling -> Classification Head     |
 |  -> 3클래스 (Real / Fake / TTS)               |
 +----------------------------------------------+
```

### 기존 1-Stage와의 핵심 차이점

| 항목 | 1-Stage (기존) | 2-Stage (제안) |
|------|---------------|---------------|
| 입력 | MFCC 13D 평균 벡터 (1D) | 106D 다중특징 시퀀스 (2D) |
| 시간 정보 | 소실 (mean pooling) | 보존 (프레임 시퀀스) |
| 특징 종류 | MFCC만 | MFCC+LFCC+Mel+Contrast+Delta |
| 모델 | FC + SE 어텐션 | Transformer / BiLSTM / Hybrid |
| 전이 학습 | 없음 | Stage 1 임베딩 재활용 |
| 데이터 증강 | SMOTE (벡터 보간) | SpecAugment (시간/특징 마스킹) |


---
## 시각화 B: 학습 과정 모니터링

학습 완료 후 실행하는 셀들이다.
학습 곡선, 혼동 행렬, ROC 커브, 모델별 성능 비교를 시각화한다.


In [ ]:
# ==============================================================
# 그래프 7: 학습 곡선 (loss / accuracy / F1) - 학습 완료 후 실행
# ==============================================================
# history 딕셔너리가 학습 셀에서 생성되어 있어야 한다.
# 구조: history[model_name] = {
#   'train_loss': [], 'val_loss': [],
#   'train_acc': [], 'val_acc': [],
#   'val_f1': []
# }

if 'history' not in dir() and 'history' not in globals():
    print('학습 완료 후 실행하세요. history 변수가 없습니다.')
else:
    n_models = len(history)
    fig, axes = plt.subplots(n_models, 3,
                             figsize=(18, 5 * n_models))
    if n_models == 1:
        axes = axes[np.newaxis, :]

    model_colors = {
        'Transformer': '#9C27B0',
        'BiLSTM': '#FF9800',
        'Hybrid': '#4CAF50'
    }

    for row, (mname, hist) in enumerate(history.items()):
        color = model_colors.get(mname, '#607D8B')
        epochs = range(1, len(hist['train_loss']) + 1)

        # Loss
        ax = axes[row, 0]
        ax.plot(epochs, hist['train_loss'], '-', color=color,
                linewidth=2, label='train')
        ax.plot(epochs, hist['val_loss'], '--', color=color,
                linewidth=2, alpha=0.7, label='val')
        best_ep = np.argmin(hist['val_loss']) + 1
        ax.axvline(best_ep, color='red', linestyle=':', alpha=0.7,
                   label=f'best ep={best_ep}')
        ax.set_title(f'{mname} - Loss')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.legend()
        ax.grid(alpha=0.3)

        # Accuracy
        ax = axes[row, 1]
        ax.plot(epochs, hist['train_acc'], '-', color=color,
                linewidth=2, label='train')
        ax.plot(epochs, hist['val_acc'], '--', color=color,
                linewidth=2, alpha=0.7, label='val')
        ax.axvline(best_ep, color='red', linestyle=':', alpha=0.7)
        ax.set_title(f'{mname} - Accuracy')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Accuracy')
        ax.set_ylim(0, 1)
        ax.legend()
        ax.grid(alpha=0.3)

        # Val F1
        ax = axes[row, 2]
        ax.plot(epochs, hist['val_f1'], '-', color=color,
                linewidth=2, label='val Macro F1')
        ax.axvline(best_ep, color='red', linestyle=':', alpha=0.7)
        best_f1 = max(hist['val_f1'])
        ax.set_title(f'{mname} - Val Macro F1\n(best: {best_f1:.4f})')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Macro F1')
        ax.set_ylim(0, 1)
        ax.legend()
        ax.grid(alpha=0.3)

    plt.suptitle('모델별 학습 곡선', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('viz_07_training_curves.png', bbox_inches='tight', dpi=120)
    plt.show()
    print('저장: viz_07_training_curves.png')


In [ ]:
# ==============================================================
# 그래프 8: 혼동 행렬 + ROC 커브 + 모델 성능 비교
# ==============================================================
# all_results 딕셔너리가 학습/평가 셀에서 생성되어 있어야 한다.

from sklearn.metrics import roc_curve, auc, classification_report

if 'all_results' not in globals():
    print('평가 완료 후 실행하세요. all_results 변수가 없습니다.')
else:
    n_models = len(all_results)
    model_colors = {
        'Transformer': '#9C27B0',
        'BiLSTM': '#FF9800',
        'Hybrid': '#4CAF50'
    }

    # (A) 혼동 행렬
    fig, axes = plt.subplots(1, n_models, figsize=(7 * n_models, 6))
    if n_models == 1:
        axes = [axes]

    for ax, (mname, res) in zip(axes, all_results.items()):
        cm = res['cm']
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
        ax.set_title(f'{mname}\n혼동 행렬 (정규화)', fontsize=11)
        ax.set_xticks(range(len(class_names)))
        ax.set_yticks(range(len(class_names)))
        ax.set_xticklabels(class_names)
        ax.set_yticklabels(class_names)
        ax.set_xlabel('예측')
        ax.set_ylabel('실제')
        plt.colorbar(im, ax=ax)
        for i in range(len(class_names)):
            for j in range(len(class_names)):
                ax.text(j, i,
                    f'{cm[i,j]}\n({cm_norm[i,j]:.2f})',
                    ha='center', va='center', fontsize=11,
                    color='white' if cm_norm[i,j] > 0.5 else 'black',
                    fontweight='bold')

    plt.suptitle('모델별 혼동 행렬', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('viz_08_confusion_matrix.png', bbox_inches='tight', dpi=120)
    plt.show()
    print('저장: viz_08_confusion_matrix.png')

    # (B) ROC 커브
    fig, axes = plt.subplots(1, n_models, figsize=(7 * n_models, 6))
    if n_models == 1:
        axes = [axes]

    for ax, (mname, res) in zip(axes, all_results.items()):
        color = model_colors.get(mname, '#607D8B')
        fpr, tpr, _ = roc_curve(res['labels'], res['probs'][:, 1])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, linewidth=2,
                label=f'{mname} (AUC={roc_auc:.4f})')
        ax.plot([0,1], [0,1], 'k--', linewidth=1, alpha=0.5)
        ax.fill_between(fpr, tpr, alpha=0.1, color=color)
        ax.set_title(f'{mname} ROC 커브')
        ax.set_xlabel('FPR (False Positive Rate)')
        ax.set_ylabel('TPR (True Positive Rate)')
        ax.legend()
        ax.grid(alpha=0.3)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1.02)

    plt.suptitle('모델별 ROC 커브', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('viz_09_roc_curve.png', bbox_inches='tight', dpi=120)
    plt.show()
    print('저장: viz_09_roc_curve.png')

    # (C) 모델 성능 비교 막대그래프
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    metrics = ['accuracy', 'macro_f1']
    metric_labels = ['Accuracy', 'Macro F1']

    for mi, (metric, label) in enumerate(zip(metrics, metric_labels)):
        ax = axes[mi]
        names = list(all_results.keys())
        values = [all_results[n][metric] for n in names]
        bar_colors = [model_colors.get(n, '#607D8B') for n in names]
        bars = ax.bar(names, values, color=bar_colors, alpha=0.85, width=0.5)
        for bar, v in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.002,
                    f'{v:.4f}', ha='center', va='bottom', fontsize=11,
                    fontweight='bold')
        ax.set_title(f'모델별 {label}')
        ax.set_ylabel(label)
        ax.set_ylim(0, 1.05)
        ax.grid(axis='y', alpha=0.3)

    # Precision / Recall / F1 클래스별 비교
    ax = axes[2]
    x = np.arange(len(class_names))
    width = 0.8 / n_models
    for mi, (mname, res) in enumerate(all_results.items()):
        report = classification_report(
            res['labels'], res['preds'],
            target_names=class_names, output_dict=True)
        f1_vals = [report[cn]['f1-score'] for cn in class_names]
        offset = (mi - n_models/2 + 0.5) * width
        ax.bar(x + offset, f1_vals, width,
               label=mname,
               color=model_colors.get(mname, '#607D8B'),
               alpha=0.85)
    ax.set_title('클래스별 F1-Score 비교')
    ax.set_xticks(x)
    ax.set_xticklabels(class_names)
    ax.set_ylabel('F1-Score')
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    plt.suptitle('모델 성능 비교', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('viz_10_model_comparison.png', bbox_inches='tight', dpi=120)
    plt.show()
    print('저장: viz_10_model_comparison.png')


In [ ]:
# ==============================================================
# 그래프 9: 특징 중요도 - 클래스간 분리도 (Fisher Score)
# ==============================================================
# Fisher Score: (클래스간 평균 차이)^2 / (클래스내 분산 합)
# 값이 높을수록 해당 특징이 real/fake 구분에 유용함

all_real = np.hstack([
    real_mfcc,
    real_mel[:, :7],   # Mel 대표 7개
    real_sc,           # Spectral Contrast 7개
    real_zcr.reshape(-1,1),
    real_rms.reshape(-1,1),
    real_sc_c.reshape(-1,1)
])  # (N, 30)

all_fake = np.hstack([
    fake_mfcc,
    fake_mel[:, :7],
    fake_sc,
    fake_zcr.reshape(-1,1),
    fake_rms.reshape(-1,1),
    fake_sc_c.reshape(-1,1)
])

feature_names = (
    [f'MFCC{i+1}' for i in range(13)] +
    [f'Mel{i+1}' for i in range(7)] +
    [f'SC{i+1}' for i in range(7)] +
    ['ZCR', 'RMS', 'Centroid']
)

n_feat = all_real.shape[1]
fisher_scores = []
for i in range(n_feat):
    mu_r = all_real[:, i].mean()
    mu_f = all_fake[:, i].mean()
    var_r = all_real[:, i].var()
    var_f = all_fake[:, i].var()
    denom = var_r + var_f + 1e-10
    fisher_scores.append((mu_r - mu_f) ** 2 / denom)

fisher_scores = np.array(fisher_scores)
sorted_idx = np.argsort(fisher_scores)[::-1]

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# 전체 Fisher Score 막대
ax = axes[0]
colors_bar = []
for name in [feature_names[i] for i in sorted_idx]:
    if 'MFCC' in name:
        colors_bar.append('#2196F3')
    elif 'Mel' in name:
        colors_bar.append('#4CAF50')
    elif 'SC' in name:
        colors_bar.append('#FF9800')
    else:
        colors_bar.append('#9C27B0')

bars = ax.barh(
    [feature_names[i] for i in sorted_idx],
    fisher_scores[sorted_idx],
    color=colors_bar, alpha=0.85
)
ax.set_title('특징별 Fisher Score (real/fake 분리도)')
ax.set_xlabel('Fisher Score (높을수록 분리 용이)')
ax.grid(axis='x', alpha=0.3)

# 범례
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2196F3', label='MFCC'),
    Patch(facecolor='#4CAF50', label='Mel-Spectrogram'),
    Patch(facecolor='#FF9800', label='Spectral Contrast'),
    Patch(facecolor='#9C27B0', label='기타 (ZCR/RMS/Centroid)')
]
ax.legend(handles=legend_elements, loc='lower right')

# 특징 그룹별 평균 Fisher Score
ax = axes[1]
groups = {
    'MFCC (1-13)': fisher_scores[:13].mean(),
    'Mel-Spec (7개)': fisher_scores[13:20].mean(),
    'Spectral Contrast': fisher_scores[20:27].mean(),
    'ZCR': fisher_scores[27],
    'RMS': fisher_scores[28],
    'Centroid': fisher_scores[29]
}
g_colors = ['#2196F3','#4CAF50','#FF9800','#9C27B0','#E91E63','#00BCD4']
bars = ax.bar(list(groups.keys()), list(groups.values()),
              color=g_colors, alpha=0.85)
for bar, v in zip(bars, groups.values()):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.001,
            f'{v:.4f}', ha='center', va='bottom', fontsize=9)
ax.set_title('특징 그룹별 평균 Fisher Score')
ax.set_ylabel('평균 Fisher Score')
ax.set_xticklabels(list(groups.keys()), rotation=15, ha='right')
ax.grid(axis='y', alpha=0.3)

plt.suptitle('음향 특징 분리도 분석 (Fisher Score)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('viz_11_fisher_score.png', bbox_inches='tight', dpi=120)
plt.show()
print('저장: viz_11_fisher_score.png')
print(f'\n분리도 TOP 5 특징:')
for i in sorted_idx[:5]:
    print(f'  {feature_names[i]:<12}: {fisher_scores[i]:.4f}')
